___
# <font color= #bbc28d> **Carga de datos a S3: maíz de PlantVillage** </font>
#### <font color= #7aa8a7> `Proyecto final · Obtención Dataset`</font>
- <Strong> Diana Valdivia </Strong> · ITESO

___

Este es el **primer paso del proyecto**: conseguir las fotos de hojas de maíz y dejarlas ordenadas en la nube, para que todo lo demás (EDA, entrenamiento y API) lea de una sola fuente.

**Qué hace, en orden:**
1. Se conecta a AWS y crea (o reutiliza) el bucket de S3.
2. Baja de GitHub **solo** las 4 carpetas de maíz de PlantVillage.
3. Les pone nombres de clase limpios y las copia a una carpeta local.
4. Las sube a S3 junto con un `class_mapping.json`.
5. Verifica que lo local y lo de S3 coincidan.

In [1]:
!pip install boto3


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


# **<font color= #7aa8a7> Configuración y permisos </font>**
Aquí se define todo lo que se usa después: la región, el nombre del bucket y el diccionario `CLASS_MAP`, que convierte los nombres largos del repositorio (`Corn_(maize)___Common_rust_`) en nombres cortos de clase (`common_rust`).

El nombre del bucket incluye el ID de la cuenta de AWS para que sea único. Al final se hace una prueba de **escribir y borrar** un archivo, para descubrir ahora (y no a la mitad de la subida) si faltan permisos.

In [4]:
import boto3, json, subprocess, shutil
from pathlib import Path
from botocore.exceptions import ClientError

# Configuración General
REGION = "us-east-1"
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
BUCKET = f"sagemaker-maize-disease-{ACCOUNT_ID}-{REGION}"
S3_PREFIX = "maize/raw"

# carpeta original del repo -> nombre limpio de clase
CLASS_MAP = {
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot": "gray_leaf_spot",
    "Corn_(maize)___Common_rust_": "common_rust",
    "Corn_(maize)___Northern_Leaf_Blight": "northern_blight",
    "Corn_(maize)___healthy": "healthy",
}

WORK = Path.home() / "maize_work"
REPO = WORK / "PlantVillage-Dataset"
CLEAN = WORK / "raw"          # copia local con nombres limpios
WORK.mkdir(exist_ok=True)

s3 = boto3.client("s3", region_name=REGION)
try:
    s3.head_bucket(Bucket=BUCKET)
    print(f"El bucket ya existe: {BUCKET}")
except ClientError:
    s3.create_bucket(Bucket=BUCKET)
    print(f"Bucket creado: {BUCKET}")

# prueba de escritura/borrado para confirmar permisos
s3.put_object(Bucket=BUCKET, Key="_test.txt", Body=b"ok")
s3.delete_object(Bucket=BUCKET, Key="_test.txt")
print("Permisos de escritura OK")

Bucket creado: sagemaker-maize-disease-248471876373-us-east-1
Permisos de escritura OK


# **<font color= #7aa8a7> Descarga parcial del repositorio </font>**
PlantVillage completo es muy pesado y trae decenas de cultivos que no necesitamos. Por eso se usa `git sparse-checkout`: se clona el repositorio **sin archivos** y luego se piden solo las 4 carpetas de maíz (`raw/color/...`)..

In [5]:
def run(cmd, cwd=None):
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

if REPO.exists():
    shutil.rmtree(REPO)

run(["git", "clone", "--filter=blob:none", "--no-checkout", "--depth", "1",
     "https://github.com/spMohanty/PlantVillage-Dataset.git", str(REPO)])
run(["git", "sparse-checkout", "init", "--cone"], cwd=REPO)
run(["git", "sparse-checkout", "set",
     *[f"raw/color/{folder}" for folder in CLASS_MAP]], cwd=REPO)
run(["git", "checkout", "master"], cwd=REPO)

$ git clone --filter=blob:none --no-checkout --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git /home/ec2-user/maize_work/PlantVillage-Dataset


Cloning into '/home/ec2-user/maize_work/PlantVillage-Dataset'...


$ git sparse-checkout init --cone
$ git sparse-checkout set raw/color/Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot raw/color/Corn_(maize)___Common_rust_ raw/color/Corn_(maize)___Northern_Leaf_Blight raw/color/Corn_(maize)___healthy
$ git checkout master
Your branch is up to date with 'origin/master'.


Already on 'master'


# **<font color= #7aa8a7> Copia local con nombres limpios </font>**
Cada imagen se copia a una carpeta con el nombre de su clase. **Se conserva el nombre original del archivo** para poder rastrear cada foto hasta el repositorio de origen.

In [6]:
if CLEAN.exists():
    shutil.rmtree(CLEAN)

EXTS = {".jpg", ".jpeg", ".png"}
local_counts = {}

for orig, clean in CLASS_MAP.items():
    src = REPO / "raw" / "color" / orig
    dst = CLEAN / clean
    dst.mkdir(parents=True, exist_ok=True)
    n = 0
    for f in src.iterdir():
        if f.suffix.lower() in EXTS:
            # se conserva el nombre original del archivo para poder rastrearlo
            shutil.copy2(f, dst / f.name)
            n += 1
    local_counts[clean] = n

print(json.dumps(local_counts, indent=2))
print("Total:", sum(local_counts.values()))

{
  "gray_leaf_spot": 513,
  "common_rust": 1192,
  "northern_blight": 985,
  "healthy": 1162
}
Total: 3852


En total son **3,852 imágenes**. El conjunto está **desbalanceado**: `common_rust` (1,192) y `healthy` (1,162) son las más grandes, mientras que `gray_leaf_spot` tiene solo 513, menos de la mitad.

Este desbalance se tendrá en cuenta más adelante: se usará **macro-F1** (que pesa igual a cada clase) y **pesos de clase** en el entrenamiento, para que la clase pequeña no quede ignorada.

# **<font color= #7aa8a7> Subida a S3 </font>**
Primero se guarda `class_mapping.json` con la relación entre nombres originales y limpios, la lista ordenada de clases y la fuente de los datos. Ese archivo después lo usa la API para saber **qué número de salida corresponde a qué clase**.

Luego `aws s3 sync` sube las imágenes a `maize/raw/{clase}/`. `sync` solo copia lo que falta, así que se puede volver a correr sin duplicar nada.

In [7]:
mapping = {
    "original_to_clean": CLASS_MAP,
    "classes": sorted(CLASS_MAP.values()),
    "source": "https://github.com/spMohanty/PlantVillage-Dataset/tree/master/raw/color",
}
(WORK / "class_mapping.json").write_text(json.dumps(mapping, indent=2, ensure_ascii=False))

run(["aws", "s3", "sync", str(CLEAN), f"s3://{BUCKET}/{S3_PREFIX}/",
     "--region", REGION, "--only-show-errors"])
run(["aws", "s3", "cp", str(WORK / "class_mapping.json"),
     f"s3://{BUCKET}/maize/class_mapping.json", "--region", REGION])

print("Subida terminada")

$ aws s3 sync /home/ec2-user/maize_work/raw s3://sagemaker-maize-disease-248471876373-us-east-1/maize/raw/ --region us-east-1 --only-show-errors
$ aws s3 cp /home/ec2-user/maize_work/class_mapping.json s3://sagemaker-maize-disease-248471876373-us-east-1/maize/class_mapping.json --region us-east-1
upload: ../maize_work/class_mapping.json to s3://sagemaker-maize-disease-248471876373-us-east-1/maize/class_mapping.json
Subida terminada


# **<font color= #7aa8a7> Verificación </font>**
Se cuentan los objetos que quedaron en S3 por clase y se comparan con el conteo local:

In [8]:
paginator = s3.get_paginator("list_objects_v2")
s3_counts = {}

for clean in CLASS_MAP.values():
    n = 0
    for page in paginator.paginate(Bucket=BUCKET, Prefix=f"{S3_PREFIX}/{clean}/"):
        n += len(page.get("Contents", []))
    s3_counts[clean] = n

print(f"{'clase':<18}{'local':>8}{'s3':>8}")
for c in sorted(CLASS_MAP.values()):
    flag = "" if local_counts[c] == s3_counts[c] else "  <-- NO COINCIDE"
    print(f"{c:<18}{local_counts[c]:>8}{s3_counts[c]:>8}{flag}")

print(f"\nRuta final: s3://{BUCKET}/{S3_PREFIX}/{{clase}}/")

clase                local      s3
common_rust           1192    1192
gray_leaf_spot         513     513
healthy               1162    1162
northern_blight        985     985

Ruta final: s3://sagemaker-maize-disease-248471876373-us-east-1/maize/raw/{clase}/
